# 💬 Case Study: Q&A Bot Over Your Own Documents

## What We're Building

A chatbot that can answer questions from **your own text files**.
Upload any `.txt` docs → ask questions → get grounded answers.

```
You: "What is the return policy?"
Bot: "According to policy.txt: items can be returned within 30 days."
```

Real use cases: internal FAQ bots, documentation assistants, customer support.

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


In [2]:
# !pip install sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


## Step 1 — Load and Chunk Your Documents

In [4]:
# Simulated documents — replace with real file reads
# e.g. text = open("my_doc.txt").read()

raw_documents = {
    "policy.txt": """
        Our return policy allows customers to return any item within 30 days of purchase.
        Items must be in original condition with tags attached.
        Refunds are processed within 5-7 business days.
        Sale items are final sale and cannot be returned.
    """,
    "shipping.txt": """
        Standard shipping takes 5-7 business days.
        Express shipping takes 2-3 business days and costs $15.
        Free shipping is available on orders over $50.
        We ship to all 50 US states.
    """,
    "hours.txt": """
        Customer support is available Monday to Friday, 9am to 6pm EST.
        Weekend support is available Saturday 10am to 4pm EST.
        We are closed on major US holidays.
    """,
}

print(f"Loaded {len(raw_documents)} documents")

Loaded 3 documents


In [5]:
# Chunk each document into sentences
# (Simple split on '.') — works well for short docs

chunks = []  # Each chunk: {"text": ..., "source": ...}

for filename, text in raw_documents.items():
    sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
    for sentence in sentences:
        chunks.append({"text": sentence, "source": filename})

print(f"Created {len(chunks)} chunks:")
for c in chunks:
    print(f"  [{c['source']}] {c['text'][:60]}")

Created 11 chunks:
  [policy.txt] Our return policy allows customers to return any item within
  [policy.txt] Items must be in original condition with tags attached
  [policy.txt] Refunds are processed within 5-7 business days
  [policy.txt] Sale items are final sale and cannot be returned
  [shipping.txt] Standard shipping takes 5-7 business days
  [shipping.txt] Express shipping takes 2-3 business days and costs $15
  [shipping.txt] Free shipping is available on orders over $50
  [shipping.txt] We ship to all 50 US states
  [hours.txt] Customer support is available Monday to Friday, 9am to 6pm E
  [hours.txt] Weekend support is available Saturday 10am to 4pm EST
  [hours.txt] We are closed on major US holidays


## Step 2 — Index (Embed) All Chunks

In [6]:
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts)

print(f"Indexed {len(embeddings)} chunks — each is a {embeddings.shape[1]}-dim vector")

Indexed 11 chunks — each is a 384-dim vector


## Step 3 — The Q&A Bot

In [7]:
def ask(question, top_k=2):
    # Embed the question
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]

    # Get top matches
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Q: {question}")
    print("Sources found:")
    for idx in top_idx:
        chunk = chunks[idx]
        print(f"  [{chunk['source']}] ({scores[idx]:.2f}) {chunk['text']}")

    # Build prompt (would send to LLM in production)
    context = " ".join(chunks[i]["text"] for i in top_idx)
    print(f"\n→ Answer would be generated from these sources")
    print("-" * 50)

ask("How long do I have to return something?")
ask("Is there free shipping?")
ask("When is customer support available on weekends?")

Q: How long do I have to return something?
Sources found:
  [policy.txt] (0.70) Our return policy allows customers to return any item within 30 days of purchase
  [policy.txt] (0.58) Refunds are processed within 5-7 business days

→ Answer would be generated from these sources
--------------------------------------------------
Q: Is there free shipping?
Sources found:
  [shipping.txt] (0.80) Free shipping is available on orders over $50
  [shipping.txt] (0.53) Express shipping takes 2-3 business days and costs $15

→ Answer would be generated from these sources
--------------------------------------------------
Q: When is customer support available on weekends?
Sources found:
  [hours.txt] (0.86) Customer support is available Monday to Friday, 9am to 6pm EST
  [hours.txt] (0.79) Weekend support is available Saturday 10am to 4pm EST

→ Answer would be generated from these sources
--------------------------------------------------


# Full Q&A bot with Claude

def ask_with_answer(question, top_k=2):
    # Embed + search
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    # Build context with source labels
    context_parts = [f"[{chunks[i]['source']}] {chunks[i]['text']}" for i in top_idx]
    context = "\n".join(context_parts)

    # Call Claude
    prompt = f"""Answer using only the information below. Cite the source file name.

{context}

Question: {question}
Answer:"""

    answer = llm(prompt, max_tokens=200)

    print(f"Q: {question}")
    print(f"A: {answer}")
    print()


ask_with_answer("How long do I have to return something?")
ask_with_answer("Is there free shipping?")
ask_with_answer("When is support available on weekends?")
